# Resume the fixed V10 research workflow

This research notebook is not a qualified portal submission. V9 remains the observed best.
Choose a free T4, run the cells in order, approve Google Drive mounting, and upload
`V10_Resume_Bundle.zip` when asked. Drive stores every completed fold directly so it
survives local app/browser interruptions. No paid resources are assumed.

The bundle preserves 26 verified fold pairs: ten V9 controls, ten C folds, and six S folds.
The six representations and four blend weights stay fixed. No V10 candidate may be
exported before the full screen and all five confirmation assignments pass.

The recorded budget amendment permits a 24-hour total cap. Prior charges are preserved;
the remaining epoch allowance is read from its ledger and may not be reset.
The code stops at its budget limits. The revised-city V7 comparison must still be fitted
on the original Mac after finalist selection; it cannot be replaced by a Linux refit.

In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,importlib.metadata
from pathlib import Path
from google.colab import drive,files
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/MLA_V10_Verified_Research')
BASE.mkdir(parents=True,exist_ok=True)
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','LOKY_MAX_CPU_COUNT']:os.environ[key]='2'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
bundle=Path('/content/V10_Resume_Bundle.zip')
if not bundle.exists():files.upload()
assert bundle.exists(),'Upload V10_Resume_Bundle.zip.'
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='f29a405408708a8537d947ca2cb3719bade4550cf7833e51f5bec005cfaa33cd','Wrong or corrupt resume bundle.'
with zipfile.ZipFile(bundle) as z:
    manifest=json.loads(z.read('resume_manifest.json'))
    for name,expected in manifest['files'].items():
        assert hashlib.sha256(z.read(name)).hexdigest()==expected,name
    existing_results=(BASE/'results').exists()
    for name in z.namelist():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        if existing_results and relative.parts[0]=='results':continue
        path=BASE/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==z.read(name),'Use a fresh Drive folder for changed source: '+name
        else:path.write_bytes(z.read(name))
print('Verified frozen sources and preserved existing Drive checkpoints:',BASE)


In [ ]:
PINS={'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'scipy': '1.18.1', 'catboost': '1.2.10', 'interpret-core': '0.7.8', 'tabpfn': '9.2.0', 'torch': '2.11.0+cu130'}
required=[name+'=='+version.split('+')[0] for name,version in PINS.items()]
subprocess.check_call([sys.executable,'-m','pip','install','-q',*required,'nbformat','nbclient'])
print('Pinned packages installed; stages run in fresh child Python processes.')


In [ ]:
environment=os.environ.copy()
environment['PYTHONDONTWRITEBYTECODE']='1'
def stage(name):
    allowance=json.loads((BASE/'results/compute_budget.json').read_text())['maximum_hours']
    argv=[sys.executable,'-u',str(BASE/'workflow.py'),name,'--data-dir',str(BASE/'data'),'--output-dir',str(BASE/'results'),'--budget-hours',str(allowance)]
    process=subprocess.Popen(argv,env=environment,cwd=BASE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:print(line,end='',flush=True)
    assert process.wait()==0,'Stage failed or budget exhausted. Preserve V9 and inspect results/stage_status.json.'
# The passing, source-verified pilot and completed fold pairs are already in the bundle.
assert json.loads((BASE/'results/pilot.json').read_text())['status']=='PASS'
stage('screen')
lock=json.loads((BASE/'results/candidate_lock.json').read_text())
print('Frozen finalist:',lock['choice'])
stage('confirm')
decision=BASE/'results/decision.json'
if decision.exists():
    stage('export')
else:
    print('City baseline pending. Do not upload a V10 CSV. Preserve V9.')
    print('Provide candidate_lock.json to your original Mac workflow, fit refit_city.py, and place its verified NPZ/JSON in this results folder before rerunning confirm.')


If the runtime stops, reconnect a free T4, mount the same Drive folder and rerun.
Existing result files are preserved and the workflow validates their source, fold,
configuration and prediction hashes. Never reset the compute ledger to bypass its limit.
Keep the one-time coding upload pending the final model choice.